In [1]:
# IMPORTS
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


Androids Analysis

In [2]:
# -----------------------------
# Paths
# -----------------------------
dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/androids_model_dataset_basic.csv")
RESULTS_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics/SVM/ANDROIDS")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Load and prepare data
# -----------------------------
df = pd.read_csv(dataset)

meta_cols = [
    "file_path", "file", "file_stem", "bdi_score",
    "depressed", "fold", "speech_type", "subgroup_from_path"
]
feature_cols = [c for c in df.columns if c not in meta_cols]

df = df.dropna(subset=feature_cols + ["depressed", "file_stem"]).copy()

X = df[feature_cols].astype(float).values
y = df["depressed"].astype(int).values
groups = df["file_stem"].astype(str).values

print("Rows:", len(df))
print("Groups:", df["file_stem"].nunique())
print("Features used:", feature_cols)

# -----------------------------
# GroupKFold CV + inner GridSearchCV
# -----------------------------
outer_cv = GroupKFold(n_splits=5)
cv_results = []

param_grid = {
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", 0.01, 0.1],
    "svm__kernel": ["rbf"]
}

for fold, (cv_train_idx, cv_val_idx) in enumerate(
    outer_cv.split(X, y, groups=groups), start=1
):
    X_train, X_val = X[cv_train_idx], X[cv_val_idx]
    y_train, y_val = y[cv_train_idx], y[cv_val_idx]
    groups_train = groups[cv_train_idx]

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(probability=True, random_state=42))
    ])

    inner_cv = GroupKFold(n_splits=3)
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=inner_cv,
        n_jobs=-1,
        refit=True
    )

    grid.fit(X_train, y_train, groups=groups_train)

    best_model = grid.best_estimator_
    preds = best_model.predict(X_val)
    proba = best_model.predict_proba(X_val)[:, 1]

    cv_results.append({
        "fold": fold,
        "n_train": len(cv_train_idx),
        "n_test": len(cv_val_idx),
        "best_params": str(grid.best_params_),
        "accuracy": accuracy_score(y_val, preds),
        "f1": f1_score(y_val, preds),
        "roc_auc": roc_auc_score(y_val, proba)
    })

cv_results_df = pd.DataFrame(cv_results)

print("\nCV fold results:")
print(cv_results_df)

cv_summary_df = pd.DataFrame([{
    "subset": "cv_train",
    "n_rows": len(df),
    "n_depressed": int(y.sum()),
    "n_control": int((y == 0).sum()),
    "accuracy_mean": cv_results_df["accuracy"].mean(),
    "accuracy_std": cv_results_df["accuracy"].std(),
    "f1_mean": cv_results_df["f1"].mean(),
    "f1_std": cv_results_df["f1"].std(),
    "roc_auc_mean": cv_results_df["roc_auc"].mean(),
    "roc_auc_std": cv_results_df["roc_auc"].std(),
}])

print("\nCV summary:")
print(cv_summary_df)

# -----------------------------
# Save results
# -----------------------------
cv_results_df.to_csv(RESULTS_PATH / "androids_svm_cv_folds.csv", index=False)
cv_summary_df.to_csv(RESULTS_PATH / "androids_svm_cv_summary.csv", index=False)

Rows: 224
Groups: 115
Features used: ['mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5', 'mfcc_6', 'mfcc_7', 'mfcc_8', 'mfcc_9', 'mfcc_10', 'mfcc_11', 'mfcc_12', 'mfcc_13', 'pitch_mean', 'energy_mean']

CV fold results:
   fold  n_train  n_test                                        best_params  \
0     1      179      45  {'svm__C': 10, 'svm__gamma': 0.1, 'svm__kernel...   
1     2      179      45  {'svm__C': 10, 'svm__gamma': 0.01, 'svm__kerne...   
2     3      179      45  {'svm__C': 0.1, 'svm__gamma': 0.01, 'svm__kern...   
3     4      179      45  {'svm__C': 0.1, 'svm__gamma': 0.01, 'svm__kern...   
4     5      180      44  {'svm__C': 10, 'svm__gamma': 0.01, 'svm__kerne...   

   accuracy        f1   roc_auc  
0  0.822222  0.870968  0.844828  
1  0.688889  0.708333  0.757937  
2  0.555556  0.714286  0.720000  
3  0.555556  0.714286  0.764000  
4  0.795455  0.769231  0.756198  

CV summary:
     subset  n_rows  n_depressed  n_control  accuracy_mean  accuracy_std  \
0  cv_train  

In [3]:
# -----------------------------
# Paths
# -----------------------------
dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/radar_model_dataset_raw_features.csv")
RESULTS_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics/SVM/RADAR")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Load and prepare data
# -----------------------------
df = pd.read_csv(dataset)

meta_cols = [
    "File", "participant_id", "Dataset", "Language", "Task",
    "recording_date", "Age", "Gender", "Education_Years",
    "Height", "phq8_score", "source_file", "depressed"
]
feature_cols = [c for c in df.columns if c not in meta_cols]

df = df.dropna(subset=feature_cols + ["phq8_score", "participant_id"]).copy()
df["depressed"] = (df["phq8_score"] >= 10).astype(int)

X = df[feature_cols].astype(float).values
y = df["depressed"].astype(int).values
groups = df["participant_id"].astype(str).values

print("Rows:", len(df))
print("Groups:", df["participant_id"].nunique())
print("Features used:", feature_cols)

# -----------------------------
# GroupKFold CV + inner GridSearchCV
# -----------------------------
outer_cv = GroupKFold(n_splits=5)
cv_results = []

param_grid = {
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", 0.01, 0.1],
    "svm__kernel": ["rbf"]
}

for fold, (cv_train_idx, cv_val_idx) in enumerate(
    outer_cv.split(X, y, groups=groups), start=1
):
    X_train, X_val = X[cv_train_idx], X[cv_val_idx]
    y_train, y_val = y[cv_train_idx], y[cv_val_idx]
    groups_train = groups[cv_train_idx]

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(probability=True, random_state=42))
    ])

    inner_cv = GroupKFold(n_splits=3)
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=inner_cv,
        n_jobs=-1,
        refit=True
    )

    grid.fit(X_train, y_train, groups=groups_train)

    best_model = grid.best_estimator_
    preds = best_model.predict(X_val)
    proba = best_model.predict_proba(X_val)[:, 1]

    cv_results.append({
        "fold": fold,
        "n_train": len(cv_train_idx),
        "n_test": len(cv_val_idx),
        "best_params": str(grid.best_params_),
        "accuracy": accuracy_score(y_val, preds),
        "f1": f1_score(y_val, preds),
        "roc_auc": roc_auc_score(y_val, proba)
    })

cv_results_df = pd.DataFrame(cv_results)

print("\nCV fold results:")
print(cv_results_df)

cv_summary_df = pd.DataFrame([{
    "subset": "cv_train",
    "n_rows": len(df),
    "n_depressed": int(y.sum()),
    "n_control": int((y == 0).sum()),
    "accuracy_mean": cv_results_df["accuracy"].mean(),
    "accuracy_std": cv_results_df["accuracy"].std(),
    "f1_mean": cv_results_df["f1"].mean(),
    "f1_std": cv_results_df["f1"].std(),
    "roc_auc_mean": cv_results_df["roc_auc"].mean(),
    "roc_auc_std": cv_results_df["roc_auc"].std(),
}])

print("\nCV summary:")
print(cv_summary_df)

# -----------------------------
# Save results
# -----------------------------
cv_results_df.to_csv(RESULTS_PATH / "radar_svm_cv_folds.csv", index=False)
cv_summary_df.to_csv(RESULTS_PATH / "radar_svm_cv_summary.csv", index=False)

Rows: 8515
Groups: 274
Features used: ['Clip_Duration', 'Speaking_Rate', 'Articulation_Rate', 'Phonation_Ratio', 'Pause_Rate', 'Pause_Ratio', 'mean_F0', 'stdev_F0_Semitone', 'HNR_dB', 'Spectral_Slope', 'Spectral_Tilt', 'Cepstral_Peak_Prominence', 'mean_F1_Loc', 'std_F1_Loc', 'mean_B1_Loc', 'std_B1_Loc', 'mean_F2_Loc', 'std_F2_Loc', 'mean_B2_Loc', 'std_B2_Loc', 'Spectral_Gravity', 'Spectral_Std_Dev']

CV fold results:
   fold  n_train  n_test                                        best_params  \
0     1     6812    1703  {'svm__C': 0.1, 'svm__gamma': 0.01, 'svm__kern...   
1     2     6812    1703  {'svm__C': 0.1, 'svm__gamma': 0.01, 'svm__kern...   
2     3     6812    1703  {'svm__C': 0.1, 'svm__gamma': 'scale', 'svm__k...   
3     4     6812    1703  {'svm__C': 0.1, 'svm__gamma': 0.01, 'svm__kern...   
4     5     6812    1703  {'svm__C': 1, 'svm__gamma': 0.01, 'svm__kernel...   

   accuracy        f1   roc_auc  
0  0.591897  0.121365  0.549236  
1  0.552554  0.199580  0.481048  
2 